# Sesión 1: Regresión en Física (Modelado de Magnitudes Continuas)
**Facilitador: Héctor Martínez | Unidad de Telecomunicaciones - ABAE**

Este cuaderno interactivo guiará los experimentos prácticos del primer bloque del taller. Abordaremos tres problemas progresivos diseñados para entender desde la unidad matemática elemental (la neurona) hasta estructuras de capas profundas aptas para aproximar funciones no lineales complejas.

## Objetivos de Aprendizaje:
1. Comprender el rol del peso ($w$) y el sesgo ($b$) mediante una relación lineal pura.
2. Identificar el impacto de las capas ocultas y funciones de activación (ReLU, softplus).
3. Diseñar una arquitectura multivariable evaluando su convergencia con curvas de pérdida (Loss Curves).

In [1]:
# Importación del ecosistema de desarrollo para IA
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

print("Versión de TensorFlow en ejecución:", tf.__version__)

Versión de TensorFlow en ejecución: 2.20.0


## 1. Calibración del Sensor de Temperatura de un Satélite (Telemetry Counts a °C)
**Propósito:** Comprender los conceptos de peso ($w$) y sesgo ($b$) aislando el ruido analítico.

Los sistemas de telemetría espacial reciben datos crudos en variables digitales conocidas como *Counts*. La relación física para convertirlos a grados Celsius responde a una calibración puramente lineal:
$$\text{Temperatura (°C)} = w \cdot \text{Counts} + b$$

Entrenaremos una sola neurona para que "descubra" la ganancia ($w$) y el offset ($b$) óptimos basándose únicamente en mediciones históricas, simulando que no poseemos la hoja de datos de calibración del fabricante.

In [ ]:
# ==========================================
# ACA EL MODELO FISICO A EMULAR!
# Ecuación real subyacente: Temp = 0.25 * Counts - 40.0

# Simulación de lecturas históricas (Relación lineal exacta oculta)
np.random.seed(42)
counts_train = np.random.uniform(0, 1023, 200) # ADC de 10 bits
temp_train = 0.25 * counts_train - 40.0
# ==========================================

# Visualización inicial del comportamiento del sensor
plt.figure(figsize=(8, 4))
plt.scatter(counts_train, temp_train, color='#38bdf8', alpha=0.6, label='Datos de Telemetría')
plt.title('Calibración Histórica del Sensor')
plt.xlabel('Counts Crudos')
plt.ylabel('Temperatura Real (°C)')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

> Observar como el cambio de 60 epocas a 200 cambia sensiblemente la aproximación del sesgo del modelo


In [ ]:
# ==========================================
# ETAPA A: DEFINICIÓN ARQUITECTÓNICA DE LA RED
# ==========================================
# Una sola capa densa con 1 nodo y entrada unidimensional
model_sensor = keras.Sequential([
    keras.Input(shape=(1,)), # Recommended way to define input shape
    keras.layers.Dense(units=1)
])

print("Resumen de la arquitectura diseñada:")
model_sensor.summary()

# ==========================================
# ETAPA B: PREPARACIÓN PARA LA COMPILACIÓN
# ==========================================
# Configuración del optimizador (Adam) y la función de pérdida para regresión (MSE)
model_sensor.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.5), # Tasa alta debido a la simplicidad del problema
    loss='mean_squared_error'
)

# Entrenamiento del modelo lineal
print("\nIniciando ajuste de calibración autónomo...")
history_sensor = model_sensor.fit(counts_train, temp_train, epochs=60, verbose=0)
print("¡Entrenamiento finalizado exitosamente!")

In [ ]:
# Visualización de la convergencia para la calibración del sensor (Sección 1)
plt.figure(figsize=(8, 4))
plt.plot(history_sensor.history['loss'], color='#f59e0b', lw=2)
plt.title('Curva de Convergencia: Calibración del Sensor')
plt.xlabel('Época de Entrenamiento')
plt.ylabel('Error Cuadrático Medio (MSE)')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

In [ ]:
# Extracción de los parámetros que la neurona aprendió de forma autónoma
pesos, sesgo = model_sensor.get_weights()
print(f"--> Parámetro w (Peso aprendido por la red): {pesos[0][0]:.4f}  (Valor real teórico: 0.25)")
print(f"--> Parámetro b (Sesgo/Offset aprendido):  {sesgo[0]:.4f} (Valor real teórico: -40.0)")

print('\nAcá el array completo:\n')
print(model_sensor.get_weights())

In [ ]:
# Generación de datos para la comparación visual
counts_test = np.linspace(0, 1023, 100).reshape(-1, 1)
temp_teorica = 0.25 * counts_test - 40.0
temp_predicha = model_sensor.predict(counts_test)

# Creación del gráfico de solapamiento
plt.figure(figsize=(10, 6))

# 1. Muestras originales de entrenamiento
plt.scatter(counts_train, temp_train, color='#38bdf8', alpha=0.3, label='Datos de Telemetría (Entrenamiento)')

# 2. Curva Teórica (La física real)
plt.plot(counts_test, temp_teorica, color='black', linestyle='--', lw=2, label='Relación Física Real (Teórica)')

# 3. Curva Predicha (Lo que aprendió la IA)
plt.plot(counts_test, temp_predicha, color='#ef4444', lw=3, label='Predicción del Modelo (IA)')

plt.title('Comparativa: Realidad Física vs. Aprendizaje del Modelo')
plt.xlabel('Counts Crudos')
plt.ylabel('Temperatura (°C)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 2. Atenuación de Señal en Telecomunicaciones por Pérdida en el Espacio Libre (FSPL)
**Propósito:** Demostrar el impacto de las capas ocultas y funciones de activación cuando la física deja de ser lineal.

La pérdida de potencia de una señal de radiofrecuencia (en dB) a medida que un satélite se desplaza respecto a la estación terrena sigue la ley de la inversa del cuadrado de la distancia. Al expresarse en escala logarítmica, el comportamiento describe una curva. Una neurona lineal simple fallará rotundamente (*underfitting*).

In [ ]:
# Simulación de pérdida de trayectoria (Curva logarítmica física)
distancia_train = np.random.uniform(400, 2000, 500) # Kilómetros (Órbita LEO)
# Ecuación física simplificada: FSPL(dB) = 20*log10(d) + Constante
path_loss_train = 20 * np.log10(distancia_train) + 32.4

plt.figure(figsize=(8, 4))
plt.scatter(distancia_train, path_loss_train, color='#f472b6', alpha=0.5, label='Muestras RF')
plt.xlabel('Distancia Satélite-Antena (km)')
plt.ylabel('Pérdida de Trayectoria (Path Loss en dB)')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

### 🧪 LABORATORIO DE ARQUITECTURAS: MODELADO DE LA ATENUACIÓN EN EL ESPACIO LIBRE (FSPL)

Para comprender de forma empírica cómo las redes neuronales profundas logran aproximar fenómenos de propagación y atenuación de ondas electromagnéticas (como la curva logarítmica de la Pérdida en el Espacio Libre - FSPL), ejecutaremos un estudio comparativo utilizando cuatro configuraciones de hardware lógico bien diferenciadas.

El objetivo es auditar cómo influyen la presencia de las **Funciones de Activación** y la **Densidad de Neuronas** en la capacidad de la red para amoldarse a la caída de potencia de la señal a lo largo de distancias kilométricas.

---

### 📋 Configuraciones del Experimento (Benchmarking de Propagación)

Mantendremos constante el número de ciclos de optimización en **`epochs = 200`** para evaluar de manera justa las siguientes cuatro arquitecturas frente a la curva de atenuación:

#### 🔹 Set 1: Red Lineal Mínima (Control de Entrada)
* **Estructura:** 1 capa oculta de solo 2 neuronas, sin funciones de activación intermedia.
* **Propósito Teórico:** Evaluar el comportamiento del modelo con la menor cantidad de recursos y sin no-linealidades. Servirá para demostrar el punto de partida más elemental de una combinación matricial rígida frente a una curva compleja.

#### 🔹 Set 2: Red Profunda Lineal (Control de Base Extendido)
* **Estructura:** 2 capas ocultas de 8 neuronas cada una, sin funciones de activación intermedia.
* **Propósito Teórico:** Demostrar empíricamente que multiplicar matrices lineales de forma sucesiva (hacer la red más profunda) sigue dando como resultado una combinación lineal rígida. Esta red se comportará exactamente igual que el Set 1 y será matemáticamente incapaz de curvarse ante la naturaleza logarítmica de la FSPL, trazando una línea recta de error.

#### 🔹 Set 3: Introducción de la No Linealidad (Aproximación por Tramos)
* **Estructura:** 2 capas ocultas de 8 neuronas, incorporando la función de activación **`ReLU`** (`Rectified Linear Unit`).
* **Propósito Teórico:** Evaluar cómo un número limitado de neuronas rectificadas intenta "romper" la rigidez lineal. Veremos cómo la activación ReLU permite a la red segmentar el espacio matemático e intentar aproximar la caída en decibelios (dB) mediante la unión de pequeños tramos rectos.

#### 🔹 Set 4: Escalado de Densidad Estructural (Suavizado Logarítmico)
* **Estructura:** Capa oculta 1 expandida a **32 neuronas**, Capa oculta 2 expandida a **16 neuronas**, ambas potenciadas con activación **`ReLU`**.
* **Propósito Teórico:** Analizar el impacto de incrementar sustancialmente el ancho y la capacidad de representación de la red. Evaluaremos si al combinar una mayor cantidad de neuronas rectificadas, la red adquiere la resolución matemática suficiente para suavizar los cambios de pendiente y calzar milimétricamente con la curva de atenuación de la señal en bruto.

---

*✍️ **Instrucciones para el Analista:** Ejecuta de forma secuencial los cuatro bloques de entrenamiento que se presentan a continuación y presta especial atención a la gráfica final superpuesta. Observa cómo el Set 1 y el Set 2 colapsan en la misma rigidez lineal, mientras que el incremento de neuronas y activaciones (Set 4) transforma los tramos toscos en una curva suave de decibelios.*

### 2.1 Probando la primera capa de activación

In [ ]:
# ==========================================
# ETAPA A: DEFINICIÓN ARQUITECTÓNICA DE LA RED
# ==========================================
model_rf = keras.Sequential([
    keras.Input(shape=(1,)), # Recommended way to define input shape
    keras.layers.Dense(units=2),
    keras.layers.Dense(units=1)
])

print("Resumen de la arquitectura diseñada:")
model_rf.summary()

# ==========================================
# ETAPA B: PREPARACIÓN PARA LA COMPILACIÓN
# ==========================================
model_rf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='mean_squared_error',
    metrics=['mean_absolute_error'] # Métrica de validación humana complementaria
)

# Ejecución del entrenamiento de la estructura profunda
print("\nIniciando ajuste de calibración autónomo...")
history_rf = model_rf.fit(distancia_train, path_loss_train, epochs=60, verbose=0)
print("¡Modelo de telecomunicaciones aproximado con éxito!")

In [ ]:
# Graficar la reducción del error a través de las iteraciones (Épocas)
plt.figure(figsize=(8, 4))
plt.plot(history_rf.history['loss'], color='#38bdf8', lw=2)
plt.title('Curva de Convergencia de la Red (Loss Curve)')
plt.xlabel('Época de Entrenamiento')
plt.ylabel('Error Cuadrático Medio (MSE)')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

In [ ]:
# Generación de predicciones para el modelo de Radiofrecuencia (Sección 2)
distancia_test = np.linspace(400, 2000, 300).reshape(-1, 1)
predicciones_rf = model_rf.predict(distancia_test)

# Visualización de resultados
plt.figure(figsize=(9, 5))
plt.scatter(distancia_train, path_loss_train, color='#f472b6', alpha=0.3, label='Muestras de Entrenamiento')
plt.plot(distancia_test, predicciones_rf, color='#0ea5e9', lw=3, label='Aproximación de la Red Neuronal')
plt.title('Modelo RF: Predicción de Pérdida en el Espacio Libre')
plt.xlabel('Distancia Satélite-Antena (km)')
plt.ylabel('Pérdida de Trayectoria (dB)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

In [ ]:
# Inspección detallada de parámetros por cada capa del modelo RF
print(f"Análisis de parámetros para: {model_rf.name}\n")
print(f'\nCapa: {model_rf.layers[0].name}:\n')
print(model_rf.layers[0].get_weights())
print(f'\nCapa: {model_rf.layers[1].name}:\n')
print(model_rf.layers[1].get_weights())

### 📐 DECONSTRUCCIÓN MATEMÁTICA: ¿QUÉ APRENDIÓ LA RED LINEAL MÍNIMA?

Para entender por qué el **Set 1** (y cualquier red profunda sin funciones de activación) es incapaz de generar una curva, vamos a abrir la "caja negra" de TensorFlow y reconstruir a mano la ecuación que dedujo el optimizador a partir de las matrices de la captura de pantalla.

Nuestra arquitectura tiene una entrada cruda $x$ (Distancia en km), conectada a una capa oculta de 2 neuronas (`dense_43`), que a su vez se conecta a una neurona de salida (`dense_44`).

---

#### 🔍 Paso 1: Operación de la Primera Capa (`dense_43`)

Al revisar la telemetría de los parámetros con `get_weights()`, observamos que la primera capa contiene:
* **Matriz de Pesos ($W_1$):** `[[1.4666417, -0.54118645]]` (Un peso para cada una de las 2 neuronas).
* **Vector de Sesgos ($b_1$):** `[-1.2982367, 0.05746816]` (Un sesgo para cada neurona).

Dado que la entrada es un único valor escalar $x$, las salidas de las dos neuronas ocultas (llamémoslas $h_1$ y $h_2$) se calculan como:

$$h_1 = 1.4666417 \cdot x - 1.2982367$$
$$h_2 = -0.54118645 \cdot x + 0.05746816$$

Cada neurona en esta etapa ha generado su propia propuesta de línea recta independiente basándose en los datos de la FSPL.

---

#### 🔍 Paso 2: Operación de la Capa de Salida (`dense_44`)

La capa final toma las salidas $h_1$ y $h_2$ como sus nuevas entradas, aplicando sus propios parámetros optimizados:
* **Matriz de Pesos ($W_2$):** `[[-0.29550597], [-0.92732704]]` (El peso que le asigna a cada neurona oculta).
* **Vector de Sesgos ($b_2$):** `[0.01712991]` (El sesgo final de la predicción).

La ecuación de salida final ($y_{pred}$) es la combinación lineal de ambas neuronas:

$$y_{pred} = (-0.29550597 \cdot h_1) + (-0.92732704 \cdot h_2) + 0.01712991$$

---

#### 🧮 Paso 3: Reducción Algebraica (Sustitución)

Si sustituimos las ecuaciones del **Paso 1** dentro de la ecuación del **Paso 2**, veremos la magia de la combinación lineal en acción:

$$y_{pred} = -0.29550597 \cdot (1.4666417x - 1.2982367) - 0.92732704 \cdot (-0.54118645x + 0.05746816) + 0.01712991$$

Al resolver las multiplicaciones distributivas respecto a $x$:

* **Componentes de la pendiente ($w_{equivalente}$):**
  $$(-0.29550597 \cdot 1.4666417)x = -0.433401 \cdot x$$
  $$(-0.92732704 \cdot -0.54118645)x = 0.501856 \cdot x$$

* **Componentes del sesgo ($b_{equivalente}$):**
  $$(-0.29550597 \cdot -1.2982367) = 0.383636$$
  $$(-0.92732704 \cdot 0.05746816) = -0.053293$$

Agrupando todos los términos semejantes:

$$y_{pred} = (-0.433401 + 0.501856) \cdot x + (0.383636 - 0.053293 + 0.01712991)$$

#### 📊 Ecuación Resultante Final:
$$\boxed{y_{pred} = 0.068455 \cdot x + 0.347473}$$

---

#### 💡 Conclusión de Ingeniería para el Análisis:

Como pueden observar, el resultado de tener dos capas densas sin función de activación se colapsó algebraicamente en **la ecuación de una línea recta simple**: una sola pendiente ($0.0684$) y un solo sesgo ($0.3474$).

**¿Qué significa esto?** Que agregar "complejidad" estructural (más capas o más neuronas) no sirve absolutamente de nada si el flujo de datos es puramente lineal. La red simplemente se dedica a sumar y multiplicar rectas, obteniendo como resultado final otra línea recta.

*Para obligar a la red a aproximar curvas reales (como la degradación logarítmica de la FSPL), es mandatorio romper esta herencia algebraica inyectando una **Función de Activación No Lineal** (como ReLU o Softplus) al final de cada capa oculta.*

### 2.2 Usando **`softplus`**

In [ ]:
# ==========================================
# ETAPA A: DEFINICIÓN ARQUITECTÓNICA DE LA RED
# ==========================================
model_rf = keras.Sequential([
    # Usamos 'softplus' para que la red maneje las distancias de miles sin morir
    keras.Input(shape=(1,)), # Recommended way to define input shape

    keras.layers.Dense(units=32, activation='softplus'),
    keras.layers.Dense(units=32, activation='softplus'),
    keras.layers.Dense(units=16, activation='softplus'),
    keras.layers.Dense(units=1)
])

print("Resumen de la arquitectura diseñada:")
model_rf.summary()

# ==========================================
# ETAPA B: PREPARACIÓN PARA LA COMPILACIÓN
# ==========================================
model_rf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='mean_squared_error',
    metrics=['mean_absolute_error'] # Métrica de validación humana complementaria
)

# Ejecución del entrenamiento de la estructura profunda
print("\nIniciando ajuste de calibración autónomo...")
history_rf = model_rf.fit(distancia_train, path_loss_train, epochs=500, verbose=0)
print("¡Modelo de telecomunicaciones aproximado con éxito!")

In [ ]:
# Graficar la reducción del error a través de las iteraciones (Épocas)
plt.figure(figsize=(8, 4))
plt.plot(history_rf.history['loss'], color='#38bdf8', lw=2)
plt.title('Curva de Convergencia de la Red (Loss Curve)')
plt.xlabel('Época de Entrenamiento')
plt.ylabel('Error Cuadrático Medio (MSE)')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

In [ ]:
# Generación de predicciones para el modelo de Radiofrecuencia (Sección 2)
distancia_test = np.linspace(400, 2000, 300).reshape(-1, 1)
predicciones_rf = model_rf.predict(distancia_test)

# Visualización de resultados
plt.figure(figsize=(9, 5))
plt.scatter(distancia_train, path_loss_train, color='#f472b6', alpha=0.3, label='Muestras de Entrenamiento')
plt.plot(distancia_test, predicciones_rf, color='#0ea5e9', lw=3, label='Aproximación de la Red Neuronal')
plt.title('Modelo RF: Predicción de Pérdida en el Espacio Libre')
plt.xlabel('Distancia Satélite-Antena (km)')
plt.ylabel('Pérdida de Trayectoria (dB)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

### problema 3: Modelado de Atenuación de Señal en Bloques de Construcción (Pérdida por Inserción Estructural)

**Contexto:** Las estaciones de referencia GNSS en tierra o los receptores de los usuarios a menudo se encuentran en entornos urbanos densos (cañones urbanos). Al intentar rastrear un satélite de navegación, la señal portadora suele atravesar estructuras y bloques de concreto. La física de telecomunicaciones nos dice que la pérdida de potencia de la señal (en dB) debido a la atenuación por inserción estructural **no es lineal**: se degrada de forma exponencial asintótica a medida que aumenta el espesor de la pared de concreto.

Si una red neuronal intenta aprender este comportamiento utilizando únicamente activaciones lineales, fallará catastróficamente, entregando una línea recta que subestima el peligro de pérdida de enlace a grosores críticos.

**Tu Objetivo:** Diseñar y entrenar una red neuronal profunda univariable que aprenda a predecir la Atenuación de la Señal (dB) basándose únicamente en el Espesor del Bloque de Concreto (metros), forzando a la red a romper la linealidad matemática para salvar el enlace de comunicaciones.

#### 📋 Guía de Implementación Paso a Paso:
1. **Fase 0 (Datos):** Ejecuta la celda del instructor para cargar las muestras experimentales (Espesor en metros vs Atenuación en dB).
2. **Etapa A (Definición Arquitectónica):** Instancia un modelo secuencial profundo para curvar el espacio matemático. Para evitar que los gradientes se queden en cero ante los valores más altos de atenuación, configura la red con la siguiente arquitectura:
   - Capa Oculta 1: `Dense` con **32 neuronas** y función de activación **`'relu'`** o **`'softplus'`**.
   - Capa Oculta 2: `Dense` con **16 neuronas** y función de activación **`'relu'`** o **`'softplus'`**.
   - Capa de Salida: **1 neurona** lineal (`units=1`, sin activación).
3. **Etapa B (Compilación Matemática):** Configura el comando `model.compile` utilizando el optimizador `Adam` (learning rate de `0.005`) y la función de pérdida idónea para regresión continua (`'mse'`).
4. **Etapa C (Entrenamiento Dinámico):** Entrena el modelo usando `model.fit` por **180 épocas**, pasando los datos de validación (`validation_data`).

In [2]:
# =====================================================================
# FASE 0: SIMULACIÓN DE ATENUACIÓN ESTRUCTURAL EN CAÑONES URBANOS
# =====================================================================
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)

# Espesor de muros de concreto simulados (de 0.05 a 1.2 metros)
X_espesor = np.random.uniform(0.05, 1.2, 200)

# Física real: Caída exponencial de potencia (Atenuación severa asintótica en dB)
# A mayor espesor, la pérdida se dispara fuertemente
Y_atenuacion_db = 3.0 + (12.0 * np.exp(2.2 * X_espesor)) + np.random.normal(0, 1.5, 200)

# Partición Train/Val (80/20)
X_train_p3, X_val_p3 = X_espesor[:160], X_espesor[160:]
Y_train_p3, Y_val_p3 = Y_atenuacion_db[:160], Y_atenuacion_db[160:]

print("--> [Fase 0] Éxito: Dataset univariable de atenuación estructural cargado en memoria.")

--> [Fase 0] Éxito: Dataset univariable de atenuación estructural cargado en memoria.
